Import libraries and path

In [17]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib

PROJECT_ROOT = Path.cwd().parent 
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

REPORTS_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODELS_DIR / "cease_risk_model.joblib"
META_PATH = MODELS_DIR / "metadata.json"

MODEL_PATH.exists(), META_PATH.exists()

(True, True)

Loading Model and metadata

In [18]:
model = joblib.load(MODEL_PATH)
with open(META_PATH, "r") as f:
    meta = json.load(f)

meta.get("model_name"), len(meta.get("features", [])), meta.keys()

('HistGradientBoostingClassifier',
 29,
 dict_keys(['trained_at', 'model_name', 'lookback_days', 'horizon_days', 'val_roc_auc_lr', 'val_pr_auc_lr', 'val_roc_auc_hgb', 'val_pr_auc_hgb', 'test_roc_auc', 'test_pr_auc', 'features', 'categorical_features', 'numeric_features', 'categorical_levels', 'feature_importance', 'top_features_for_ui', 'top_call_types']))

Loading Validation and test sets

In [19]:
val_df = pd.read_parquet(INTERIM_DIR / "val.parquet")
test_df = pd.read_parquet(INTERIM_DIR / "test.parquet")

val_df.shape, test_df.shape

((296982, 28), (261737, 28))

Checking columns and spellings

In [20]:
TARGET = "target_30d"
ID_COL = "unique_customer_identifier"
DATE_COL = "snapshot_date"

feature_cols = meta["features"]
cat_cols = meta.get("categorical_features", [])

required_cols = set([TARGET, ID_COL, DATE_COL]) | set(feature_cols)

missing_val = sorted(required_cols - set(val_df.columns))
missing_test = sorted(required_cols - set(test_df.columns))

print("Missing in val:", missing_val[:20], "count:", len(missing_val))
print("Missing in test:", missing_test[:20], "count:", len(missing_test))

# We'll add log features next; so allow missing log1p_* for now
non_log_missing_val = [c for c in missing_val if not c.startswith("log1p_")]
non_log_missing_test = [c for c in missing_test if not c.startswith("log1p_")]

assert len(non_log_missing_val) == 0, "Validation is missing non-log required columns"
assert len(non_log_missing_test) == 0, "Test is missing non-log required columns"

Missing in val: ['log1p_avg_hold_time_30d', 'log1p_avg_talk_time_30d', 'log1p_sum_download_30d', 'log1p_sum_upload_30d'] count: 4
Missing in test: ['log1p_avg_hold_time_30d', 'log1p_avg_talk_time_30d', 'log1p_sum_download_30d', 'log1p_sum_upload_30d'] count: 4


Standardise categoricals and add log features

In [21]:
def standardise_categoricals(df: pd.DataFrame, cat_cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].astype(str).fillna("Unknown").str.strip()
    return df

def add_log_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ["sum_download_30d", "sum_upload_30d", "avg_talk_time_30d", "avg_hold_time_30d"]:
        if col in df.columns:
            df[f"log1p_{col}"] = np.log1p(pd.to_numeric(df[col], errors="coerce").fillna(0).clip(lower=0))
    return df

val_df = standardise_categoricals(val_df, cat_cols)
test_df = standardise_categoricals(test_df, cat_cols)

val_df = add_log_features(val_df)
test_df = add_log_features(test_df)

Verify  all model features exist

In [9]:
val_proba = model.predict_proba(val_df[feature_cols])[:, 1]
test_proba = model.predict_proba(test_df[feature_cols])[:, 1]

val_scored = val_df[[ID_COL, DATE_COL, TARGET]].copy()
val_scored["risk_score"] = val_proba

test_scored = test_df[[ID_COL, DATE_COL, TARGET]].copy()
test_scored["risk_score"] = test_proba

val_scored.head()

,unique_customer_identifier,snapshot_date,target_30d,risk_score
0,2147b7e439b1e4dd597e88ee7cf15a4fc23d27046e7e17...,2024-05-01,0,0.083137
1,214fce5d267b45ba497710c529863fc1c9e733c3255beb...,2024-06-01,1,0.844723
2,21562ee19c61b797d76598c8be6fa2a492ceb840f52938...,2024-05-01,0,0.073508
3,21581257132d92a65c95b626d57dc452e717f6ed94e49f...,2024-06-01,0,0.273790
4,215b291bcfeb532e61e9444947bc23f55350f644f29efe...,2024-05-01,0,0.184566


Score val/test

In [22]:
val_proba = model.predict_proba(val_df[feature_cols])[:, 1]
test_proba = model.predict_proba(test_df[feature_cols])[:, 1]

val_scored = val_df[[ID_COL, DATE_COL, TARGET]].copy()
val_scored["risk_score"] = val_proba

test_scored = test_df[[ID_COL, DATE_COL, TARGET]].copy()
test_scored["risk_score"] = test_proba

val_scored.head()

,unique_customer_identifier,snapshot_date,target_30d,risk_score
0,c5d3ec9a466948152428ff03cdfc29aa07fc4076c04272...,2024-06-01,0,0.063374
1,c5e0e59ee2345e644218e3b7bc4a41a7541640729b5ec1...,2024-06-01,0,0.057416
2,c5e161d08373171d1c3668a2212077b2fc348036c07793...,2024-05-01,1,0.976573
3,c5e71467a668dbd08bed1a540700f76921eb125528ca5a...,2024-05-01,0,0.165853
4,c5e978c9061315ea90d6312d67bd52bc26db4e23d73ce5...,2024-04-01,0,0.143620


Customer Segmentation

In [23]:
def add_decile(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # decile 10 = highest risk
    df["risk_decile"] = pd.qcut(df["risk_score"], 10, labels=False, duplicates="drop") + 1
    return df

val_scored = add_decile(val_scored)
test_scored = add_decile(test_scored)

overall_rate_val = val_scored[TARGET].mean()
overall_rate_test = test_scored[TARGET].mean()

decile_table_val = (
    val_scored.groupby("risk_decile")
    .agg(n=("risk_score","size"), cease_rate=(TARGET,"mean"), avg_score=("risk_score","mean"))
    .assign(lift=lambda d: d["cease_rate"] / overall_rate_val)
    .sort_index(ascending=False)
)

decile_table_test = (
    test_scored.groupby("risk_decile")
    .agg(n=("risk_score","size"), cease_rate=(TARGET,"mean"), avg_score=("risk_score","mean"))
    .assign(lift=lambda d: d["cease_rate"] / overall_rate_test)
    .sort_index(ascending=False)
)

decile_table_val, decile_table_test

(                 n  cease_rate  avg_score      lift
 risk_decile                                        
 10           29699    0.334759   0.861100  7.092119
 9            29698    0.080308   0.484671  1.701395
 8            29698    0.021988   0.244043  0.465833
 7            29698    0.009967   0.158845  0.211159
 6            29698    0.008182   0.113134  0.173350
 5            29698    0.006330   0.084685  0.134114
 4            29697    0.004647   0.068429  0.098449
 3            29699    0.002795   0.053970  0.059208
 2            29698    0.002121   0.037357  0.044943
 1            29699    0.000909   0.022433  0.019260,
                  n  cease_rate  avg_score      lift
 risk_decile                                        
 10           26172    0.242396   0.858596  7.068983
 9            26111    0.055302   0.471100  1.612777
 8            26234    0.016696   0.238782  0.486901
 7            26178    0.008786   0.160590  0.256225
 6            26173    0.006342   0.117555  0

Capacity-based bands

In [25]:
CALL_PCT = 0.05          # capacity for outbound calls (top 5%)
MEDIUM_TOP_PCT = 0.20    # next 15% gets low-cost actions

q_high = float(np.quantile(val_scored["risk_score"], 1 - CALL_PCT))
q_medium = float(np.quantile(val_scored["risk_score"], 1 - MEDIUM_TOP_PCT))

def assign_band(score: float) -> str:
    if score >= q_high:
        return "High"
    elif score >= q_medium:
        return "Medium"
    return "Low"

val_scored["risk_band"] = val_scored["risk_score"].apply(assign_band)
test_scored["risk_band"] = test_scored["risk_score"].apply(assign_band)

val_scored["risk_band"].value_counts(normalize=True), q_high, q_medium

(risk_band
 Low       0.799998
 Medium    0.149999
 High      0.050003
 Name: proportion, dtype: float64,
 0.8692974452427101,
 0.3129284470076412)

Band Performance table

In [26]:
def band_summary(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby("risk_band")
        .agg(
            n_customers=("risk_score", "size"),
            avg_score=("risk_score", "mean"),
            cease_rate=(TARGET, "mean"),
        )
        .sort_values("avg_score", ascending=False)
    )

band_summary_val = band_summary(val_scored)
band_summary_test = band_summary(test_scored)

band_summary_val, band_summary_test

(           n_customers  avg_score  cease_rate
 risk_band                                    
 High             14850   0.933433    0.440741
 Medium           44547   0.586034    0.129795
 Low             237585   0.097862    0.007117,
            n_customers  avg_score  cease_rate
 risk_band                                    
 High             13177   0.933382    0.315929
 Medium           37647   0.585031    0.095227
 Low             210913   0.099747    0.005818)

Call list KPIs (precision/recall at capacity)

In [27]:
def call_list_kpis(df: pd.DataFrame) -> tuple[int, float, float]:
    called = df[df["risk_band"] == "High"]
    precision = float(called[TARGET].mean())                   # hit rate among called
    recall = float(called[TARGET].sum() / df[TARGET].sum())    # cease capture
    return len(called), precision, recall

n_call_val, prec_val, rec_val = call_list_kpis(val_scored)
n_call_test, prec_test, rec_test = call_list_kpis(test_scored)

print("VAL  High band:", n_call_val, "precision:", round(prec_val,3), "recall:", round(rec_val,3))
print("TEST High band:", n_call_test, "precision:", round(prec_test,3), "recall:", round(rec_test,3))

VAL  High band: 14850 precision: 0.441 recall: 0.467
TEST High band: 13177 precision: 0.316 recall: 0.464


Interventions

In [28]:
# Bring key features into the scored frames (only what we need for reasons)
reason_features = [c for c in [
    "dd_cancel_60_day", "contract_dd_cancels", "ooc_days",
    "calls_30d", "avg_download_30d", "sum_download_30d",
    "contract_status"
] if c in val_df.columns]

val_enriched = val_scored.join(val_df[reason_features])
test_enriched = test_scored.join(test_df[reason_features])

# thresholds from validation
calls_hi = float(np.quantile(val_enriched["calls_30d"], 0.90)) if "calls_30d" in val_enriched else 0
usage_lo = float(np.quantile(val_enriched["avg_download_30d"], 0.10)) if "avg_download_30d" in val_enriched else 0

def reason_code(row):
    # Payment disruption
    if ("dd_cancel_60_day" in row and row["dd_cancel_60_day"] == 1) or ("contract_dd_cancels" in row and row["contract_dd_cancels"] > 0):
        return "Payment disruption"
    # Out of contract / nearing end
    if "ooc_days" in row and row["ooc_days"] > 0:
        return "Out of contract (OOC)"
    # Low usage (possible disengagement / service issue)
    if "avg_download_30d" in row and row["avg_download_30d"] <= usage_lo:
        return "Low usage / disengagement"
    # High call volume
    if "calls_30d" in row and row["calls_30d"] >= calls_hi:
        return "High contact volume"
    return "General risk"

val_enriched["primary_reason"] = val_enriched.apply(reason_code, axis=1)
test_enriched["primary_reason"] = test_enriched.apply(reason_code, axis=1)

val_enriched["primary_reason"].value_counts().head(10)

primary_reason
High contact volume          147571
Out of contract (OOC)        118633
Low usage / disengagement     19388
Payment disruption            11390
Name: count, dtype: int64

Recommended action

In [29]:
def action_plan(band, reason):
    if band == "High":
        if reason == "Payment disruption":
            return "Urgent outbound call (48h): billing support + direct debit rescue + retention offer"
        if reason == "Out of contract (OOC)":
            return "Urgent outbound call (48h): contract renewal discussion + tailored offer"
        if reason == "High contact volume":
            return "Urgent outbound call (48h): resolve service issues + goodwill credit if appropriate"
        if reason == "Low usage / disengagement":
            return "Outbound call (48h): service health check + engagement offer"
        return "Outbound call (48h): retention specialist + tailored offer/support"

    if band == "Medium":
        if reason == "Payment disruption":
            return "Proactive SMS/email: payment support + link to billing help; escalate if no response"
        if reason == "Out of contract (OOC)":
            return "Email/SMS: renewal reminder + online offer; schedule call if risk increases"
        if reason == "High contact volume":
            return "Email/SMS: troubleshooting/self-serve + option to book support call"
        return "Low-cost intervention: targeted email/SMS + monitor next month"

    return "No proactive outreach; monitor monthly; include in general retention comms"

val_enriched["recommended_action"] = val_enriched.apply(lambda r: action_plan(r["risk_band"], r["primary_reason"]), axis=1)
test_enriched["recommended_action"] = test_enriched.apply(lambda r: action_plan(r["risk_band"], r["primary_reason"]), axis=1)

val_enriched[["risk_score","risk_band","primary_reason","recommended_action"]].head()

,risk_score,risk_band,primary_reason,recommended_action
0,0.063374,Low,High contact volume,No proactive outreach; monitor monthly; includ...
1,0.057416,Low,High contact volume,No proactive outreach; monitor monthly; includ...
2,0.976573,High,Payment disruption,Urgent outbound call (48h): billing support + ...
3,0.165853,Low,Out of contract (OOC),No proactive outreach; monitor monthly; includ...
4,0.143620,Low,High contact volume,No proactive outreach; monitor monthly; includ...


Export operational call list (High risk only)

In [15]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

call_list_val = (
    val_scored[val_scored["risk_band"] == "High"]
    .sort_values("risk_score", ascending=False)
    .reset_index(drop=True)
)

out_csv = REPORTS_DIR / "call_list_val_top5pct.csv"
call_list_val.to_csv(out_csv, index=False)

out_csv, call_list_val.head(10)

(WindowsPath('c:/Users/icons/Documents/UKTelecomCeasePrediction/reports/call_list_val_top5pct.csv'),
                           unique_customer_identifier snapshot_date  \
 0  a2f6025d7c7069cf1c2a040d2ceacd874ce16747c18409...    2024-05-01   
 1  7132b32dcf29676c5a074a96a5807b9126bebf2e0a49e9...    2024-05-01   
 2  93dfb7064b8f231522549f3721ddb650646a3663cdf85b...    2024-04-01   
 3  e8006542a2f987550387cd6379a71eff646ab877155312...    2024-06-01   
 4  8ff6a017385ed5057b4172c7624f3bbfa2f6552b0a1115...    2024-06-01   
 5  04f7948c0990be80bcdee3bf78f67189f9c77026b638f2...    2024-04-01   
 6  e32f94e1710938ad6b72d2ece85137900be7002a2c3388...    2024-06-01   
 7  732aff01f6164e7c686a7d975fa1a155071006058e9fbe...    2024-05-01   
 8  54b5cfdcf45a77d64c560c190aa8ad7c60e7aa630553b9...    2024-04-01   
 9  c7e02819167b31bfacbd95055fb6e766cb5d2143147923...    2024-05-01   
 
    target_30d  risk_score risk_band  \
 0           0    0.989289      High   
 1           1    0.988103      High

Save thresholds and business KPIs

In [30]:
meta["risk_band_policy"] = {"high_top_pct": CALL_PCT, "medium_top_pct": MEDIUM_TOP_PCT}
meta["band_thresholds_from_val"] = {"q_high": q_high, "q_medium": q_medium}

meta["business_kpis_val"] = {"call_list_size": int(n_call_val), "precision": prec_val, "recall": rec_val}
meta["business_kpis_test"] = {"call_list_size": int(n_call_test), "precision": prec_test, "recall": rec_test}

# Add decile lift summaries for slides
meta["decile_lift_val_top_decile"] = float(decile_table_val.loc[10, "lift"]) if 10 in decile_table_val.index else None
meta["decile_lift_test_top_decile"] = float(decile_table_test.loc[10, "lift"]) if 10 in decile_table_test.index else None

with open(META_PATH, "w") as f:
    json.dump(meta, f, indent=2)

print("Updated metadata:", META_PATH)

Updated metadata: c:\Users\icons\Documents\UKTelecomCeasePrediction\models\metadata.json


Save slide-ready tables

In [31]:
decile_table_val.to_csv(REPORTS_DIR / "decile_lift_val.csv")
decile_table_test.to_csv(REPORTS_DIR / "decile_lift_test.csv")
band_summary_val.to_csv(REPORTS_DIR / "band_summary_val.csv")
band_summary_test.to_csv(REPORTS_DIR / "band_summary_test.csv")

print("Saved reporting tables into reports/")

Saved reporting tables into reports/


Life and Capture

In [32]:
id="kpi_lift_capture"
overall_val = val_scored["target_30d"].mean()
overall_test = test_scored["target_30d"].mean()

def add_kpis(scored_df, overall_rate):
    # cease captured by each band
    total_ceases = scored_df["target_30d"].sum()
    out = (
        scored_df.groupby("risk_band")
        .agg(
            n_customers=("risk_score","size"),
            cease_rate=("target_30d","mean"),
            ceases=("target_30d","sum")
        )
        .assign(
            lift=lambda d: d["cease_rate"] / overall_rate,
            capture=lambda d: d["ceases"] / total_ceases
        )
        .sort_values("cease_rate", ascending=False)
    )
    return out, float(total_ceases), float(overall_rate)

kpi_val, total_ceases_val, base_val = add_kpis(val_scored, overall_val)
kpi_test, total_ceases_test, base_test = add_kpis(test_scored, overall_test)

print("VAL baseline cease rate:", round(base_val,4), "total ceases:", int(total_ceases_val))
display(kpi_val)

print("TEST baseline cease rate:", round(base_test,4), "total ceases:", int(total_ceases_test))
display(kpi_test)

VAL baseline cease rate: 0.0472 total ceases: 14018


,n_customers,cease_rate,ceases,lift,capture
risk_band,,,,,
High,14850,0.440741,6545,9.337428,0.466900
Medium,44547,0.129795,5782,2.749816,0.412470
Low,237585,0.007117,1691,0.150789,0.120631


TEST baseline cease rate: 0.0343 total ceases: 8975


,n_customers,cease_rate,ceases,lift,capture
risk_band,,,,,
High,13177,0.315929,4163,9.213413,0.463844
Medium,37647,0.095227,3585,2.777087,0.399443
Low,210913,0.005818,1227,0.169657,0.136713


Call Capabilities

In [36]:
import pandas as pd
import numpy as np

def capacity_metrics(scored_df, call_pct, overall_rate):
    q = float(np.quantile(scored_df["risk_score"], 1 - call_pct))
    called = scored_df[scored_df["risk_score"] >= q]
    precision = float(called["target_30d"].mean())
    recall = float(called["target_30d"].sum() / scored_df["target_30d"].sum())
    lift = precision / overall_rate
    return {
        "call_pct": call_pct,
        "n_called": int(len(called)),
        "precision": precision,
        "recall": recall,
        "lift": lift
    }

overall_val = float(val_scored["target_30d"].mean())
overall_test = float(test_scored["target_30d"].mean())

rows_val = [capacity_metrics(val_scored, p, overall_val) for p in [0.03, 0.05, 0.10]]
rows_test = [capacity_metrics(test_scored, p, overall_test) for p in [0.03, 0.05, 0.10]]

pd.DataFrame(rows_val), pd.DataFrame(rows_test)

(   call_pct  n_called  precision    recall       lift
 0      0.03      8910   0.504153  0.320445  10.680857
 1      0.05     14850   0.440741  0.466900   9.337428
 2      0.10     29699   0.334759  0.709231   7.092119,
    call_pct  n_called  precision    recall       lift
 0      0.03      7853   0.365847  0.320111  10.669171
 1      0.05     13088   0.317084  0.462396   9.247098
 2      0.10     26175   0.242407  0.706964   7.069287)

Update metadata

In [39]:
# Save updated metadata (thresholds + KPIs)
meta["band_thresholds_from_val"] = {"q_high": float(q_high), "q_medium": float(q_medium)}
meta["risk_band_policy"] = {"high_top_pct": 0.05, "medium_top_pct": 0.20}

meta["kpis_table_val"] = kpi_val.reset_index().to_dict(orient="records")
meta["kpis_table_test"] = kpi_test.reset_index().to_dict(orient="records")

with open(META_PATH, "w") as f:
    json.dump(meta, f, indent=2)

# Export call list
call_list_val = (val_enriched[val_enriched["risk_band"]=="High"]
                 .sort_values("risk_score", ascending=False)
                 .reset_index(drop=True))

out_csv = REPORTS_DIR / "call_list_val_top5pct.csv"
call_list_val.to_csv(out_csv, index=False)

out_csv

WindowsPath('c:/Users/icons/Documents/UKTelecomCeasePrediction/reports/call_list_val_top5pct.csv')

In [38]:
drivers = [c for c in [
    "dd_cancel_60_day", "contract_dd_cancels", "ooc_days",
    "calls_30d", "avg_download_30d", "contract_status"
] if c in val_df.columns]

val_enriched = val_scored.join(val_df[drivers])
test_enriched = test_scored.join(test_df[drivers])

calls_hi = float(np.quantile(val_enriched["calls_30d"], 0.90)) if "calls_30d" in val_enriched else 0
usage_lo = float(np.quantile(val_enriched["avg_download_30d"], 0.10)) if "avg_download_30d" in val_enriched else 0

def reason_code(row):
    if ("dd_cancel_60_day" in row and row["dd_cancel_60_day"] == 1) or ("contract_dd_cancels" in row and row["contract_dd_cancels"] > 0):
        return "Payment disruption"
    if "ooc_days" in row and row["ooc_days"] > 0:
        return "Out of contract (OOC)"
    if "avg_download_30d" in row and row["avg_download_30d"] <= usage_lo:
        return "Low usage / disengagement"
    if "calls_30d" in row and row["calls_30d"] >= calls_hi:
        return "High contact volume"
    return "General risk"

def action_plan(band, reason):
    if band == "High":
        if reason == "Payment disruption":
            return "Urgent call (48h): billing support + DD recovery + retention offer"
        if reason == "Out of contract (OOC)":
            return "Urgent call (48h): renewal discussion + tailored offer"
        if reason == "High contact volume":
            return "Urgent call (48h): resolve issues + goodwill credit if needed"
        if reason == "Low usage / disengagement":
            return "Call (48h): service health check + engagement offer"
        return "Urgent call (48h): retention specialist + tailored support"
    if band == "Medium":
        if reason == "Payment disruption":
            return "SMS/email: billing help + payment link; escalate if no response"
        if reason == "Out of contract (OOC)":
            return "Email/SMS: renewal reminder + online offer; monitor"
        if reason == "High contact volume":
            return "Email/SMS: troubleshooting resources + option to book support"
        return "Targeted email/SMS + monitor"
    return "No outreach; monitor monthly; general comms"

val_enriched["primary_reason"] = val_enriched.apply(reason_code, axis=1)
test_enriched["primary_reason"] = test_enriched.apply(reason_code, axis=1)

val_enriched["recommended_action"] = val_enriched.apply(lambda r: action_plan(r["risk_band"], r["primary_reason"]), axis=1)
test_enriched["recommended_action"] = test_enriched.apply(lambda r: action_plan(r["risk_band"], r["primary_reason"]), axis=1)

val_enriched[["risk_score","risk_band","primary_reason","recommended_action"]].head(10)

,risk_score,risk_band,primary_reason,recommended_action
0,0.063374,Low,High contact volume,No outreach; monitor monthly; general comms
1,0.057416,Low,High contact volume,No outreach; monitor monthly; general comms
2,0.976573,High,Payment disruption,Urgent call (48h): billing support + DD recove...
3,0.165853,Low,Out of contract (OOC),No outreach; monitor monthly; general comms
4,0.143620,Low,High contact volume,No outreach; monitor monthly; general comms
5,0.077569,Low,High contact volume,No outreach; monitor monthly; general comms
6,0.108245,Low,Out of contract (OOC),No outreach; monitor monthly; general comms
7,0.206789,Low,Out of contract (OOC),No outreach; monitor monthly; general comms
8,0.044180,Low,High contact volume,No outreach; monitor monthly; general comms
9,0.246463,Low,Low usage / disengagement,No outreach; monitor monthly; general comms


In [40]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd().parent
META_PATH = PROJECT_ROOT / "models" / "metadata.json"

with open(META_PATH, "r") as f:
    meta = json.load(f)

model_features = meta["features"][:]  # keep as-is for inference
cat_cols = meta.get("categorical_features", [])
feature_importance = meta.get("feature_importance", {})
top_ui = meta.get("top_features_for_ui", [])

# 1) define features that should NOT appear in the UI (but keep for model)
DROP_FROM_UI = {"null_score", "rn", "calls_nan_30d"}
# you can also hide crm_package_name if you want a cleaner single form:
# DROP_FROM_UI.add("crm_package_name")

# 2) create a UI-safe ordered list (use your existing top_features_for_ui first)
ui_features_ordered = [f for f in top_ui if f not in DROP_FROM_UI]

# If list is short, backfill from feature importance
if len(ui_features_ordered) < 12 and feature_importance:
    ranked = sorted(feature_importance.items(), key=lambda kv: kv[1], reverse=True)
    for f, _ in ranked:
        if f.startswith("log1p_"):
            continue
        if f in DROP_FROM_UI:
            continue
        if f in ui_features_ordered:
            continue
        ui_features_ordered.append(f)
        if len(ui_features_ordered) >= 12:
            break

# 3) ensure categorical dropdown values are stable (Unknown first)
categorical_levels = meta.get("categorical_levels", {})
for c in cat_cols:
    vals = categorical_levels.get(c, [])
    if not vals:
        continue
    vals = [str(v).strip() for v in vals if str(v).strip()]
    # put Unknown first if present
    if "Unknown" in vals:
        vals = ["Unknown"] + [v for v in vals if v != "Unknown"]
    categorical_levels[c] = vals

meta["categorical_levels"] = categorical_levels

# 4) clean top_call_types (remove nan-like entries)
top_call_types = meta.get("top_call_types", [])
top_call_types_clean = []
for t in top_call_types:
    t_str = str(t).strip()
    if t_str.lower() in {"nan", "none", ""}:
        continue
    top_call_types_clean.append(t_str)

meta["top_call_types_clean"] = top_call_types_clean

# 5) add explicit keys the Streamlit app should use
meta["model_features"] = model_features                 # for inference
meta["ui_features_for_form"] = ui_features_ordered      # for single prediction form
meta["ui_drop_features"] = sorted(list(DROP_FROM_UI))   # for transparency

# 6) optional: flag that some features are engineered/dedup artifacts
meta["notes"] = meta.get("notes", {})
meta["notes"]["feature_hygiene"] = (
    "model_features must not be changed without retraining. "
    "ui_features_for_form is safe to change for Streamlit display."
)

with open(META_PATH, "w") as f:
    json.dump(meta, f, indent=2)

print("Updated metadata saved:", META_PATH)
print("UI features (single form):", meta["ui_features_for_form"])
print("Top call types (clean):", meta["top_call_types_clean"])

Updated metadata saved: c:\Users\icons\Documents\UKTelecomCeasePrediction\models\metadata.json
UI features (single form): ['ooc_days', 'contract_dd_cancels', 'dd_cancel_60_day', 'line_speed', 'tenure_days', 'speed', 'avg_download_30d', 'calls_Loyalty_30d', 'sum_download_30d', 'avg_upload_30d', 'sales_channel', 'calls_30d']
Top call types (clean): ['Tech', 'CS&B', 'Loyalty', 'Customer Finance', 'FTTP']
